In [1]:
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import FunctionTransformer
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import torch.optim as optim
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, classification_report
from scipy.signal import butter, sosfilt

from torchmetrics import Accuracy, Precision, Recall, F1Score, ConfusionMatrix

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DCASE2024_ROOT_PATH = Path.cwd().parent / "data/raw/dcase-2024"

DCASE2024_TRAIN_PATH = DCASE2024_ROOT_PATH / "Train"
DCASE2024_DEV_PATH = DCASE2024_ROOT_PATH / "Dev"
DCASE2024_EVAL_PATH = DCASE2024_ROOT_PATH / "Eval"

DCASE2022_ROOT_PATH = Path.cwd().parent / "data/raw/dcase-2022"

DCASE2022_TRAIN_PATH = DCASE2022_ROOT_PATH / "Train"
DCASE2022_DEV_PATH = DCASE2022_ROOT_PATH / "Dev"

# ===============================================================
# ===============================================================

SAMPLE_RATE = 16_000
MEL_SPECTROGRAM_PARAMS = {
    "n_fft": 1024,
    "hop_length": 512,
    "n_mels": 128,
    "window": "hann",
    "center": True,
    "pad_mode": "reflect",
    "power": 2.0,
}

MFCC_PARAMS = {
    "n_mfcc": 128,
    "n_fft": 1024,
    "hop_length": 512,
    "n_mels": 128,
    "dct_type": 2,
    "norm": "ortho",
    "lifter": 0,
}

device

device(type='cuda')

In [3]:
import librosa
import numpy as np


def get_mel_spectrogram(signal, params):
    mel = librosa.feature.melspectrogram(y=signal, **params)

    log_mel = librosa.power_to_db(mel, ref=np.max)

    return log_mel.astype(np.float32)


def get_mfcc(signal, params):
    mfcc = librosa.feature.mfcc(y=signal, **params)

    return mfcc.astype(np.float32)

In [4]:
def load_audio(audio):
    signal, sample_rate = librosa.load(audio, mono=True, sr=SAMPLE_RATE)

    return signal, sample_rate

def pad_or_trim(signal, target_len):
    if len(signal) > target_len:
        return signal[:target_len]
    elif len(signal) < target_len:
        return np.pad(signal, (0, target_len - len(signal)), 'constant')

    return signal

def rms_normalize(signal, target_rms=0.1, eps=1e-8):
    rms = np.sqrt(np.mean(signal**2) + eps)
    gain = target_rms / (rms + eps)

    return (signal * gain).astype(np.float32)

def peak_normalize(signal):
    peak = np.max(np.abs(signal))
    if peak > 0:
        return signal / peak
    return signal


def preprocess_data(signals, padding_or_trim=True, denoise_method=None, normalize_method=None):
    mel_features = []
    mfcc_features = []
    target_len = 10 * SAMPLE_RATE

    for signal in signals:
        if padding_or_trim:
            signal = pad_or_trim(signal, target_len)
        if denoise_method:
            signal = denoise_method(signal)
        if normalize_method:
            signal = normalize_method(signal)

        mel = get_mel_spectrogram(signal, MEL_SPECTROGRAM_PARAMS)
        mfcc = get_mfcc(signal, MFCC_PARAMS)

        # norm_mel = min_max_scaler(mel)
        # norm_mfcc = min_max_scaler(mfcc)

        mel_features.append(mel)
        mfcc_features.append(mfcc)

    return mel_features, mfcc_features

def bandpass_filter(signal, sr=16000, low=80, high=7500, order=4):
    nyq = 0.5 * sr
    low_n = low / nyq
    high_n = high / nyq
    sos = butter(order, [low_n, high_n], btype="bandpass", output="sos")
    return sosfilt(sos, signal).astype(np.float32)

def spectral_gate(signal, sr=16000, noise_sec=0.25):
    n = int(noise_sec * sr)
    noise_clip = signal[:n]
    return nr.reduce_noise(y=signal, y_noise=noise_clip, sr=sr).astype(np.float32)

def highpass_filter(signal, sr=16000, cutoff=80, order=4):
    nyq = 0.5 * sr
    cutoff_n = cutoff / nyq
    sos = butter(order, cutoff_n, btype="highpass", output="sos")
    return sosfilt(sos, signal).astype(np.float32)

# Leitura dos áudios


In [5]:
def read_dcase_dev_set(path, ignore_machine_types=None, only_machine_type=None):
    if ignore_machine_types is None:
        ignore_machine_types = []

    X_train, y_train, mtype_train = [], [], []
    X_test, y_test, mtype_test = [], [], []

    for machine_dir in Path(path).iterdir():
        if not machine_dir.is_dir():
            continue

        mtype = machine_dir.name
        if mtype in ignore_machine_types:
            continue

        if only_machine_type is not None and mtype != only_machine_type:
            continue

        for section in ["train", "test"]:
            data_dir = machine_dir / section
            if not data_dir.exists():
                continue

            wavs = list(data_dir.glob("*.wav"))
            for audio_file in tqdm(wavs, desc=f"{mtype} - {section}"):
                try:
                    signal, _ = load_audio(str(audio_file))

                    if section == "train":
                        label = 0
                        X_train.append(signal)
                        y_train.append(label)
                        mtype_train.append(mtype)

                    else:
                        label = 1 if "anomaly" in audio_file.name.lower() else 0
                        X_test.append(signal)
                        y_test.append(label)
                        mtype_test.append(mtype)

                except Exception as e:
                    print(f"Erro ao processar {audio_file.name}: {e}")

    return X_train, X_test, y_train, y_test, mtype_train, mtype_test


def read_dcase_train_set(path):
    X = []
    machine_type = []
    y = []

    for machine_dir in Path(path).iterdir():
        if not machine_dir.is_dir():
            continue

        data_dir = machine_dir / "train"
        if not data_dir.exists():
            continue

        for audio_file in tqdm(data_dir.iterdir(), desc="Lendo train"):
            if audio_file.suffix != ".wav":
                continue
            try:
                label = 0

                signal, _ = load_audio(audio_file)

                X.append(signal)
                machine_type.append(machine_dir.name)
                y.append(label)

            except Exception as e:
                print(f"Erro ao processar {audio_file.name}: {e}")

    return X, machine_type, y


def read_dcase_eval_set(path):
    X = []
    machine_type = []
    y = []

    for machine_dir in Path(path).iterdir():
        if not machine_dir.is_dir():
            continue

        data_dir = machine_dir / "test"
        if not data_dir.exists():
            continue

        for audio_file in tqdm(data_dir.iterdir(), desc="Lendo eval"):
            if audio_file.suffix != ".wav":
                continue
            try:
                if "anomaly" in audio_file.name:
                    label = 1
                else:
                    label = 0

                signal, _ = load_audio(audio_file)

                X.append(signal)
                machine_type.append(machine_dir.name)
                y.append(label)

            except Exception as e:
                print(f"Erro ao processar {audio_file.name}: {e}")

    return X, machine_type, y

In [6]:
def read_dcase_dev_set(path, ignore_machine_types=None, only_machine_type=None, only_section=None, ignore_target_domain=True):
    if ignore_machine_types is None:
        ignore_machine_types = []

    X_train, y_train, mtype_train, section_train = [], [], [], []
    X_test, y_test, mtype_test, section_test = [], [], [], []

    for machine_dir in Path(path).iterdir():
        if not machine_dir.is_dir():
            continue

        mtype = machine_dir.name
        if mtype in ignore_machine_types:
            continue

        if only_machine_type is not None and mtype != only_machine_type:
            continue

        for section_folder in ["train", "test"]:
            data_dir = machine_dir / section_folder
            if not data_dir.exists():
                continue

            wavs = list(data_dir.glob("*.wav"))
            for audio_file in tqdm(wavs, desc=f"{mtype} - {section_folder}"):

                # --- NOVA VERIFICAÇÃO: IGNORAR TARGET DOMAIN ---
                if ignore_target_domain and "target" in audio_file.name.lower():
                    continue

                try:
                    # Extrai a seção do nome do arquivo
                    parts = audio_file.name.split('_')
                    if 'section' in parts:
                        idx = parts.index('section')
                        section_id = f"section_{parts[idx+1]}"
                    else:
                        section_id = "unknown"

                    # Se você especificou uma seção e ela for diferente da atual, pula este áudio
                    if only_section is not None and section_id != only_section:
                        continue

                    signal, _ = load_audio(str(audio_file))

                    if section_folder == "train":
                        label = 0
                        X_train.append(signal)
                        y_train.append(label)
                        mtype_train.append(mtype)
                        section_train.append(section_id)

                    else:
                        label = 1 if "anomaly" in audio_file.name.lower() else 0
                        X_test.append(signal)
                        y_test.append(label)
                        mtype_test.append(mtype)
                        section_test.append(section_id)

                except Exception as e:
                    print(f"Erro ao processar {audio_file.name}: {e}")

    return X_train, X_test, y_train, y_test, mtype_train, mtype_test, section_train, section_test

In [ ]:
signals_train, signals_test, labels_train, labels_test, mtype_train, mtype_test, sec_train, sec_test = read_dcase_dev_set(
    DCASE2022_DEV_PATH,
    only_machine_type='gearbox',
    only_section='section_00'
)
print()
print('=' * 45)
print('Leitura Concluída!')
print(f'Total de amostras: {len(signals_train) + len(signals_test)}')
print(f'Amostras de treino: {len(signals_train)}')
print(f'Amostras de testes: {len(signals_test)}')
print(f'Máquinas de treino: {np.unique(mtype_train)}')
print(f'Máquinas de teste: {np.unique(mtype_test)}')
print('=' * 45)
print()

fan - train:   0%|          | 0/3000 [00:00<?, ?it/s]c:\Users\josel\OneDrive\Desktop\NCIA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
fan - test: 100%|██████████| 600/600 [00:06<00:00, 94.17it/s] 


Leitura Concluída!
Total de amostras: 3270
Amostras de treino: 2970
Amostras de testes: 300
Máquinas de treino: ['fan']
Máquinas de teste: ['fan']



In [8]:
class MachineAudioDataset(Dataset):
    def __init__(self, features, labels):
       self.features = features
       self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feature = self.features[idx]
        label = self.labels[idx]

        return (torch.tensor(feature, dtype=torch.float32),
                torch.tensor(label, dtype=torch.long))

In [9]:
def to_time_major(feat_ct: np.ndarray) -> np.ndarray:
    return feat_ct.T.astype(np.float32)

def fit_zscore(train_feats_tc: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    all_frames = np.concatenate(train_feats_tc, axis=0)  # (sum_T, C)
    mean = all_frames.mean(axis=0).astype(np.float32)
    std = (all_frames.std(axis=0) + 1e-8).astype(np.float32)
    return mean, std

def apply_zscore(feats_tc: list[np.ndarray], mean: np.ndarray, std: np.ndarray) -> list[np.ndarray]:
    out = []
    for x in feats_tc:
        out.append(((x - mean) / std).astype(np.float32))
    return out

In [10]:
import torch
import torch.nn as nn

class CNNAutoencoder(nn.Module):
    def __init__(self):
        super(CNNAutoencoder, self).__init__()
        
        # ENCODER: Compressão Espacial
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
        )
        
        # GARGALO REAL (BOTTLENECK): Força a rede a comprimir tudo em 256 valores
        self.fc1 = nn.Linear(128 * 20 * 8, 256) 
        self.fc2 = nn.Linear(256, 128 * 20 * 8)

        # DECODER: Reconstrução
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1), 
            nn.BatchNorm2d(16), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),
        )

    def forward(self, x):
        x = x.unsqueeze(1) 
        original_shape = x.shape
        
        # Passa pela CNN
        encoded = self.encoder(x)
        conv_shape = encoded.shape 
        
        # Achata e passa pelo Gargalo
        encoded_flat = encoded.view(encoded.size(0), -1)
        bottleneck = torch.relu(self.fc1(encoded_flat))
        decoded_flat = torch.relu(self.fc2(bottleneck))
        
        # Desachata e passa pelo Decoder
        decoded_reshaped = decoded_flat.view(encoded.size(0), conv_shape[1], conv_shape[2], conv_shape[3])
        decoded = self.decoder(decoded_reshaped)
        
        decoded = decoded[:, :, :original_shape[2], :original_shape[3]]
        return decoded.squeeze(1)

In [11]:
sec_unique = sorted(list(set(sec_train)))
sec_to_idx = {s:i for i,s in enumerate(sec_unique)}
y_train_section = np.array([sec_to_idx[s] for s in sec_train], dtype=np.int64)

y_test_anom = np.array(labels_test, dtype=np.int64)  

train_mel, train_mfcc = preprocess_data(signals_train, denoise_method=None, normalize_method=peak_normalize)
test_mel,  test_mfcc  = preprocess_data(signals_test,  denoise_method=None, normalize_method=peak_normalize)

train_mel = [to_time_major(m) for m in train_mel]  # (T, C)
test_mel  = [to_time_major(m) for m in test_mel]

mean, std = fit_zscore(train_mel)
train_mel = [apply_zscore(x, mean, std) for x in train_mel]
test_mel  = [apply_zscore(x, mean, std) for x in test_mel]

train_features = train_mel
test_features = test_mel

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SectionCNN(nn.Module):
    def __init__(self, n_sections: int, embed_dim: int = 256):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_emb = nn.Linear(256, embed_dim)
        self.fc_cls = nn.Linear(embed_dim, n_sections)

    def forward(self, x):
        # x: (B, T, C)
        x = x.unsqueeze(1)  # (B, 1, T, C)
        z = self.net(x)
        z = self.pool(z).flatten(1)
        emb = self.fc_emb(z)
        emb = F.normalize(emb, dim=1)
        logits = self.fc_cls(emb)
        return logits, emb

In [13]:
from torch.utils.data import Dataset, DataLoader

train_ds = MachineAudioDataset(train_features, y_train_section)
test_ds  = MachineAudioDataset(test_features, y_test_anom)  # labels de anomalia só para AUC

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

In [ ]:
n_sections = len(sec_unique)
model = SectionCNN(n_sections=n_sections, embed_dim=256).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

epochs = 25
for epoch in range(1, epochs + 1):
    model.train()
    losses = []
    correct = 0
    total = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        opt.zero_grad(set_to_none=True)
        logits, _ = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        opt.step()

        losses.append(loss.item())
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()

    print(f"epoch {epoch:02d} loss {np.mean(losses):.4f} acc {correct/max(total,1):.3f}")

C:\Users\josel\AppData\Local\Temp\ipykernel_27552\3065367794.py:13: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  return (torch.tensor(feature, dtype=torch.float32),


In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def get_embeddings(loader):
    model.eval()
    embs = []
    labels = []
    for x, y in loader:
        x = x.to(device)
        _logits, emb = model(x)
        embs.append(emb.cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(embs, axis=0), np.concatenate(labels, axis=0)

train_embs, _ = get_embeddings(train_loader)     # só normal
test_embs, y_test = get_embeddings(test_loader)  # y_test é 0 normal, 1 anomaly

knn = NearestNeighbors(n_neighbors=5, metric="cosine")
knn.fit(train_embs)

dists, _ = knn.kneighbors(test_embs)
scores = dists.mean(axis=1)

auc = roc_auc_score(y_test, scores)
print("AUC:", auc)

AUC: 0.6086222222222222
